In [ ]:
import scanpy as sc
import pandas as pd
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
#import scrublet as scr
import tqdm.auto as tqdm
import seaborn as sns
import gseapy as gp
import ktplotspy as kpy
import networkx as nx
from bioinfokit.visuz import GeneExpression

import cellphonedb
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method
from cellphonedb.src.core.methods import cpdb_analysis_method
from cellphonedb.utils import search_utils

import itertools
from mousipy import translate
import harmonypy

In [ ]:
import h5py
with h5py.File('/data/bonn-epyc/projects/manuela/aging/aging_muscle/soupxed_samples/rgzj1_soupx.h5ad', 'r') as f:
    print(dict(f.attrs))

In [ ]:
%matplotlib  inline

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
!grep -r "scrublet" ~/

# Read in Data

In [ ]:
# Define paths to your data
data_paths = ['/data/bonn-epyc/projects/manuela/aging/aging_muscle/soupxed_samples/rgzj1_soupx.h5ad',
              '/data/bonn-epyc/projects/manuela/aging/aging_muscle/soupxed_samples/rgzj2_soupx.h5ad',
              '/data/bonn-epyc/projects/manuela/aging/aging_muscle/soupxed_samples/rgzj3_soupx.h5ad',
              '/data/bonn-epyc/projects/manuela/aging/aging_muscle/soupxed_samples/rgzj4_soupx.h5ad',
              '/data/bonn-epyc/projects/manuela/aging/aging_muscle/soupxed_samples/rgzj5_soupx.h5ad']

# Concatenate the datasets
adata = ad.concat({file:sc.read_h5ad(file) for file in data_paths})
adata.obs = adata.obs.rename(columns={'orig.ident': 'sample'})

# Ensure unique cell and gene names
adata.var_names_make_unique()
adata.obs_names_make_unique()
adata.var_names

In [ ]:
value_mapping = {
    'SG18': 'ctrl',
    'SG20': 'age',
    'SG24': 'sAct',
    'SG26': 'DR',
    'SG28': 'combi',
    'sg18': 'ctrl',
    'sg20': 'age',
    'sg24': 'sAct',
    'sg26': 'DR',
    'sg28': 'combi',
    'rgzj1': 'ctrl',
    'rgzj2': 'age',
    'rgzj3': 'sAct',
    'rgzj4': 'DR',
    'rgzj5': 'combi'
}

adata.obs['sample'].replace(value_mapping, inplace=True)

# Scrublet

In [ ]:
results = pd.read_csv("/data/bonn-epyc/projects/manuela/aging/aging_muscle/scrublet-aging_muscle_soupxed.csv", index_col = 0)

In [ ]:
adata.obs['sample'].value_counts()

In [ ]:
#Check number of doublets
adata.obs['doublets'] = results['predicted_doublets']
adata[adata.obs['doublets']==True]

In [ ]:
#filter out predicted doublets
adata = adata[adata.obs['doublets']==False]

print(adata.obs['sample'].value_counts())
adata[adata.obs['doublets']==True]

# Basic preprocessing

In [ ]:
# List of marker genes
marker_genes_mouse = ['Pdpn', 'Lyve1', 'Prox1', 'Flt4']

# Check if marker genes are present in the Anndata object
present_marker_genes = [gene for gene in marker_genes_mouse if gene in adata.var_names]

# Print the list of marker genes present in the Anndata object
print("Marker genes present in the Anndata object:", present_marker_genes)

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20, 
                         # save='aging_highest_expr_genes.png'
                        )

In [ ]:
sc.pp.filter_cells(adata, min_genes=400)
sc.pp.filter_genes(adata, min_cells=3)

adata.var['mt'] = adata.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], jitter =0.4, multi_panel= True, groupby = 'sample', rotation = 90)

sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
# instead of picking subjectively, you can use a quantile
upper_lim = np.quantile(adata.obs.n_genes_by_counts.values, .98)
lower_lim = np.quantile(adata.obs.n_genes_by_counts.values, .02)
print(f'{lower_lim} to {upper_lim}')

#adata = adata[(adata.obs.n_genes_by_counts < upper_lim) & (adata.obs.n_genes_by_counts > lower_lim)]

In [ ]:
adata = adata[adata.obs.n_genes < 5000, :]
adata = adata[adata.obs.pct_counts_mt < 5, :]

In [ ]:
# print The total number of nuclei retained after QC filtering HERE, after all filters are applied
print(adata.obs['sample'].value_counts())
print(f'Total nuclei retained: {adata.n_obs}')

In [ ]:
# Normalize, log-transform and scale the data
sc.pp.normalize_total(adata, target_sum=1e4) #normalize every cell to 10000 UMI (10000 counts)
print(adata.X[0,:].sum()) #sum up all counts for cell 0 -> sums will be 10000 now

sc.pp.log1p(adata) #change to log counts

## Clustering

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)
#adata.var[adata.var.highly_variable]

In [ ]:
# Choose the gene to visualize
gene_of_interest = 'Myh4'  # Replace with the actual gene name

# Perform UMAP on the subset of highly variable genes
sc.pp.neighbors(adata)
sc.tl.umap(adata)

# Plot gene expression on UMAP
plt.figure(figsize=(6, 4))
sc.pl.umap(adata, color=[gene_of_interest], size=50)
#plt.title(f'Expression of {gene_of_interest}')
plt.show()

In [ ]:
# List of genes to include
genes_to_include = ['Ttn', 
                    'Myh4'
                   ]  # Replace with the actual gene names

# Set the 'highly_variable' attribute to True for each gene
for gene in genes_to_include:
    adata.var.loc[gene, 'highly_variable'] = True

In [ ]:
target_gene = "Myh4"  # Replace with the actual gene name you're looking for

# Check if the gene is in the AnnData object's variable (gene) names
if target_gene in adata.var_names:
    print(f"{target_gene} is present in the AnnData object.")
else:
    print(f"{target_gene} is not present in the AnnData object.")
    
gene_name = 'Ttn'

if gene_name in adata.var_names:
    print(f"{gene_name} is present in the dataset.")

# Save adata.raw

In [ ]:
adata.raw = adata # new slot adata.raw to save data before processing

# Filter highly variable genes
adata = adata[:, adata.var.highly_variable]
adata.shape

In [ ]:
print(adata.obs)

# Step 1: Extract count matrix
count_matrix = adata.X

# Step 2: Extract gene names
gene_names = adata.var_names

# Step 3: Extract condition labels
condition_labels = adata.obs['sample']

# Define the gene of interest
gene_of_interest = "Ptprd"

# Initialize a dictionary to store counts by condition
counts_by_condition = {}

# Step 4: Iterate over condition labels and count expression of the gene of interest
for condition in set(condition_labels):
    # Filter count matrix based on condition
    indices = condition_labels == condition
    counts_for_condition = count_matrix[indices, :]
    
    # Find index of the gene of interest
    gene_index = list(gene_names).index(gene_of_interest)
    
    # Sum the counts of the gene for the current condition
    counts_for_gene = counts_for_condition[:, gene_index].sum()
    
    # Store the counts for the current condition
    counts_by_condition[condition] = counts_for_gene

# Print counts by condition
for condition, count in counts_by_condition.items():
    print(f"Gene {gene_of_interest} counts in {condition}: {count}")

In [ ]:
# Regress out effects of total counts per cell and the percentage of mitochondrial genes expressed. 
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

# Scale the data to unit variance and zero mean
sc.pp.scale(adata, max_value=10)

# Perform PCA
sc.tl.pca(adata, svd_solver='arpack', use_highly_variable=True)
sc.pl.pca(adata, color = 'sample')
sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
print(adata.obs)

# Step 1: Extract count matrix
count_matrix = adata.X

# Step 2: Extract gene names
gene_names = adata.var_names

# Step 3: Extract condition labels
condition_labels = adata.obs['sample']

# Define the gene of interest
gene_of_interest = "Ptprd"

# Initialize a dictionary to store counts by condition
counts_by_condition = {}

# Step 4: Iterate over condition labels and count expression of the gene of interest
for condition in set(condition_labels):
    # Filter count matrix based on condition
    indices = condition_labels == condition
    counts_for_condition = count_matrix[indices, :]
    
    # Find index of the gene of interest
    gene_index = list(gene_names).index(gene_of_interest)
    
    # Sum the counts of the gene for the current condition
    counts_for_gene = counts_for_condition[:, gene_index].sum()
    
    # Store the counts for the current condition
    counts_by_condition[condition] = counts_for_gene

# Print counts by condition
for condition, count in counts_by_condition.items():
    print(f"Gene {gene_of_interest} counts in {condition}: {count}")

In [ ]:
print(adata.shape)
print(adata.obs.shape)
print(adata.obsm['X_pca'].shape)

In [ ]:
# Run Harmony
harmony_out = harmonypy.run_harmony(
    adata.obsm['X_pca'], 
    adata.obs, 
    'sample'
)

# Store directly into internal dict, bypassing anndata validation
adata.obsm._data['X_pca_harmony'] = np.array(harmony_out.Z_corr)

# Verify
print(adata.obsm['X_pca_harmony'].shape)  # should show (24419, 50)

In [ ]:
# Calculate neighborhood graph
#sc.pp.neighbors(adata, n_neighbors=10, n_pcs=20, use_rep="X_pca_harmony")
sc.pp.neighbors(adata, n_pcs=20, metric="cosine", n_neighbors=30, use_rep="X_pca_harmony")

# Clustering
sc.tl.leiden(adata, resolution=0.31)

# Compute UMAP
sc.tl.umap(adata)

In [ ]:
# Visualization
with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.umap(adata, color=['sample', 'leiden',], wspace=0.4, save = 'Umap_aging.png')    
    sc.pl.umap(adata, color=['n_genes_by_counts', 'total_counts', 'nFeature_RNA'], wspace=0.4)   

#sc.pl.umap(adata, color=['sample'], figsize=(4,6))
#sc.pl.umap(adata, color=['leiden'])
#sc.pl.umap(adata, color=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'])

In [ ]:
print(adata.obs['sample'].value_counts())
print(f'Total nuclei retained: {adata.n_obs}')

# Output integrated object

In [ ]:
#adata.var[adata.var.index == 'Ptprd']
#adata.var
#adata.obs

In [ ]:
adata.raw.var[adata.raw.var.index == 'Lsamp']

In [ ]:
# 1. Identify the target genes
target_genes = ['Ptprd']

# 2. Extract the expression values for the target genes
target_gene_expr = adata.raw[:, target_genes].X

# 3. Group the expression values by 'sample' and sum them
sample_labels = adata.obs['sample']

sample_target_gene_expr = pd.Series(target_gene_expr.tocsr().sum(axis=1).A1, index=sample_labels).groupby(level=0).sum()
sample_target_gene_expr

## Output Subclusters

In [ ]:
sub12 = adata[adata.obs.leiden=='12']
sc.pp.neighbors(sub12, n_pcs=20, metric="cosine", n_neighbors=30, use_rep="X_pca_harmony")
sc.tl.leiden(sub12, key_added='subclusters', resolution=0.31)

fig = sc.pl.umap(sub12, color=['subclusters'], title= "subclutser of cluster 12",show = False)

outsub12 = '/data/bonn-epyc/projects/manuela/scRNA_aging-mouse/sub12_integrated-aging-soupxed.h5ad'
sub12.write(outsub12)

In [ ]:
sub5 = adata[adata.obs.leiden=='5']
sc.pp.neighbors(sub5, n_pcs=20, metric="cosine", n_neighbors=30, use_rep="X_pca_harmony")
sc.tl.leiden(sub5, key_added='subclusters', resolution=0.31)

fig = sc.pl.umap(sub5, color=['subclusters'], title= "subclutser of cluster 5",show = False)

outsub5 = '/data/bonn-epyc/projects/manuela/scRNA_aging-mouse/sub5_integrated-aging-soupxed.h5ad'
sub5.write(outsub5)

## Dendogram

In [ ]:
sc.tl.dendrogram(adata, groupby='sample')
sc.pl.dendrogram(adata, 'sample')
#print(adata.uns['dendrogram_sample'])
sc.pl.correlation_matrix(adata, 'sample')

### lymphatic marker gene

In [ ]:
histopath_genes = ['Myl4', 'Sparc', 'Hspg2', 'Vim', 'Fn1','Thbs4', 'Bgn', 'Ctsk', 'Spp1']

# Visualize expression of the lymphatic marker gene per Leiden cluster
sc.pl.dotplot(
    adata,
    var_names=histopath_genes,  # List containing the lymphatic marker gene
    groupby='leiden',  # Group by the Leiden clusters
    use_raw=True,  # Use preprocessed data or raw data
    standard_scale='var',  # Standardize the expression for the gene
    figsize=(5, 6),  # Adjust the figure size
)

# Official Marker Gene List

In [ ]:
# Define the marker genes and subtypes along with their corresponding cell types
marker_dict = {
    'All Muscle Nuclei': ['Ttn'],
    'Neuromuscular Junction (NMJ)': ['Ufsp1', 'Ache', 'Phldb2', 'Chrna1', 'Chrnb', 'Etv4', 'Etv5'],
    #'Neuromuscular Junction': ['Ank3', 'Ablim2', 'Utrn', 'Ncam1', 'Col19a1', 'Chrne', 'Cped1', 'Lama2'],
    'Myotendinous Junction (MTJ)': ['Tigd4', 'Col22a1', 'Sorbs2', 'Lama2', 'Peli1', 'Med12l', 'Ankrd1'],
    'Myh1+ General Myonuclei': ['Myh1'],
    'Myh4+ General Myonuclei': ['Myh4'],
    'Non-canonical MTJ (rare myonuclei)': ['Ebf1', 'Col6a1', 'Col6a3', 'Abca8'],
    'Rian (rare myonuclei)': ['Rian', 'Meg3'],
    'Perimysium (rare myonuclei)': ['Muc13']
}

# Filter the marker dictionary to only include genes in adata.var_names
filtered_marker_dict = {cell_type: [gene for gene in genes if gene in adata.raw.var_names]
                        for cell_type, genes in marker_dict.items() if any(gene in adata.var_names for gene in genes)}

# Print the expression dictionary (you can format or visualize as needed)
print(filtered_marker_dict)

In [ ]:
#mp = 
sc.pl.matrixplot(adata, filtered_marker_dict, groupby='leiden', use_raw=True)
#mp.add_totals().style(edge_color='black').show()

#mp2 = sc.pl.matrixplot(adata, filtered_marker_dict, groupby='leiden', use_raw=False, return_fig=True)
#mp2.add_totals().style(edge_color='black').show()

In [ ]:
# Convert marker_dict to a pandas DataFrame
df = pd.DataFrame(marker_dict.items(), columns=['Cell Type', 'Genes'])
# Set display option to show all entries in the 'Genes' column
pd.set_option('display.max_colwidth', None)

df

## Dotplot

In [ ]:
sc.pl.dotplot(adata, filtered_marker_dict, groupby='leiden', standard_scale="var")

#sc.pl.dotplot(adata[adata.obs['sample'] == 'rgzj1'], filtered_marker_dict, groupby='leiden', standard_scale="var" )
#sc.pl.dotplot(adata[adata.obs['sample'] == 'rgzj2'], filtered_marker_dict, groupby='leiden',  standard_scale="var")
#sc.pl.dotplot(adata[adata.obs['sample'] == 'rgzj3'], filtered_marker_dict, groupby='leiden', standard_scale="var")
#sc.pl.dotplot(adata[adata.obs['sample'] == 'rgzj4'], filtered_marker_dict, groupby='leiden',  standard_scale="var")
#sc.pl.dotplot(adata[adata.obs['sample'] == 'rgzj5'], filtered_marker_dict, groupby='leiden',  standard_scale="var")

## UMAPs

In [ ]:
for ct in filtered_marker_dict:
   # print(f"{ct.upper()}:")  # print cell subtype name
    print(ct) 
    sc.pl.umap(
        adata,
        color=filtered_marker_dict[ct],
        vmin=0,
        vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
        sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
        frameon=False,
        cmap="Reds",  # or choose another color map e.g. from here: https://matplotlib.org/stable/tutorials/colors/colormaps.html
      #  save = '_'+ct
    )
    print("\n\n\n")  # print white space for 

## Marker Gene expression comparison between samples

In [ ]:
sc.pl.dotplot(adata, filtered_marker_dict, groupby='sample', standard_scale="var")

## Cluster expansion across samples

In [ ]:
# Assuming your clusters are stored in the 'leiden' column
clusters_to_explore = [4, 12]

# Create an empty DataFrame to store the results
cluster_expansion_data = []

# Loop through each sample
for sample in adata.obs['sample'].unique():
    # Filter the data for the current sample
    subset_adata = adata[adata.obs['sample'] == sample]
    
    # Calculate the proportions of cells in each cluster
    cluster_counts = subset_adata.obs['leiden'].value_counts(normalize=True)
    
    # Extract the proportions for the clusters of interest
    proportions = [cluster_counts.get(cluster, 0) for cluster in clusters_to_explore]
    
    cluster_expansion_data.append({
        'Sample': sample,
        **{f'Cluster_{cluster}': proportion for cluster, proportion in zip(clusters_to_explore, proportions)}
    })

# Create a DataFrame from the collected data
cluster_expansion_df = pd.DataFrame(cluster_expansion_data)

# Plotting
for cluster in clusters_to_explore:
    plt.figure(figsize=(10, 6))
    plt.bar(cluster_expansion_df['Sample'], cluster_expansion_df[f'Cluster_{cluster}'], color='skyblue')
    plt.title(f'Cluster {cluster} Expansion in Samples')
    plt.xlabel('Sample')
    plt.ylabel('Proportion')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
# Calculate cell type proportions in each sample
ct_sample_counts = pd.crosstab(adata.obs['leiden'], adata.obs['sample'])
ct_sample_proportions = ct_sample_counts.div(ct_sample_counts.sum(axis=1), axis=0)

# Create a heatmap of cell type proportions
plt.figure(figsize=(10, 10))
sns.heatmap(
    ct_sample_proportions,
    cmap='YlGnBu',  # Choose your preferred colormap
    annot=True, fmt=".2f",
    xticklabels=True,
    yticklabels=True,
    cbar_kws={'label': 'Cell Type Proportion'},
)
plt.title('Subtype Proportions in Samples')
plt.ylabel('Cluster')
plt.xlabel('Sample')
plt.tight_layout()
plt.show()

# DEGs btw. Clusters

In [ ]:
# Rank genes
sc.tl.rank_genes_groups(adata, 'leiden', method='t-test')

# Plot the ranked genes
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)
#Dotplot top genes
sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden", standard_scale="var", n_genes=5, title='Top 5 Genes expression in clusters', key="rank_genes_groups")

In [ ]:
# Rank genes using the t-test method and store the results in 'adata.uns' (if not done already)
sc.tl.rank_genes_groups(adata, 'leiden', method='t-test')

# Extract the results from 'adata.uns' and store them in a dataframe
result_dict = adata.uns['rank_genes_groups']

# Initialize empty lists to store the results for each cluster
cluster_list = []
gene_list = []
score_list = []
p_value_list = []
padj_value_list = []
lfc_list = [] 

# Iterate over the clusters and extract the results
for cluster in result_dict['names'].dtype.names:
    genes = result_dict['names'][cluster]
    scores = result_dict['scores'][cluster]
    p_values = result_dict['pvals'][cluster]
    padj_values = result_dict['pvals_adj'][cluster]
    lfc_values = result_dict['logfoldchanges'][cluster]
    
    # Append the results to the lists
    cluster_list.extend([cluster] * len(genes))
    gene_list.extend(genes)
    score_list.extend(scores)
    p_value_list.extend(p_values)
    padj_value_list.extend(padj_values)
    lfc_list.extend(lfc_values)

# Create the DataFrame
result_df = pd.DataFrame({
    'cluster': cluster_list,
    'gene': gene_list,
    'score': score_list,
    'p_value': p_value_list,
    'padj_value': padj_value_list,
    'lfc': lfc_list,
})

# Print the dataframe
print(result_df[result_df['p_value'] != 0])

In [ ]:
#Filter results
result_df[(result_df.padj_value < 0.05) & (abs(result_df.lfc)>0)]

In [ ]:
#One specific cluster
result_df[result_df.cluster == '5']

In [ ]:
# label all cells with an individual gene
ttn_i = np.where(adata.raw.var_names == 'Ttn')[0][0]

#get counts?
glis = adata.raw.X.toarray()[:, ttn_i]

adata.obs['ttn'] = glis
adata.obs[adata.obs['ttn'] > 0]

plt.rcParams['figure.figsize'] = (10, 5) 

sc.pl.violin(adata, ['Ttn'], groupby='leiden')
sc.pl.violin(adata, ['Ttn'], groupby='sample')

#Reset the default figure size after plotting
plt.rcParams['figure.figsize'] = [6.4, 4.8]  # Reset to the default size

In [ ]:
# label all cells with an individual gene
myh4_i = np.where(adata.raw.var_names == 'Myh4')[0][0]

#get counts?
glis = adata.raw.X.toarray()[:, myh4_i]

adata.obs['myh4'] = glis
adata.obs[adata.obs['myh4'] > 0]

plt.rcParams['figure.figsize'] = (10, 5) 

sc.pl.violin(adata, ['Myh4'], groupby='leiden')
sc.pl.violin(adata, ['Myh4'], groupby='sample')

#Reset the default figure size after plotting
plt.rcParams['figure.figsize'] = [6.4, 4.8]  # Reset to the default size

## Top DEGs

In [ ]:
# Get top 10 marker genes for each cluster
marker_genes = {}
for group in adata.uns['rank_genes_groups']['names'].dtype.names:
    marker_genes[group] = adata.uns['rank_genes_groups']['names'][group][:20]

# Convert marker_genes to pandas DataFrame for easy visualization
marker_genes_df = pd.DataFrame(marker_genes)
#print(marker_genes_df)

In [ ]:
marker_genes_dict = marker_genes_df.to_dict(orient='list')
#print(marker_genes_dict)

In [ ]:
# Initialize an empty dictionary to store common genes by cell type
common_genes_by_celltype = {}

# Iterate through the cell types and initialize lists for common genes
for cell_type in filtered_marker_dict:
    common_genes_by_celltype[cell_type] = []

# Iterate through the clusters and their DE gene lists
for cluster, marker_genes_list in marker_genes_dict.items():
    marker_genes_set = set(marker_genes_list)
    
    # Iterate through the cell types and their marker gene lists
    for cell_type, filtered_marker_genes in filtered_marker_dict.items():
        filtered_marker_genes_set = set(filtered_marker_genes)
        common_genes = marker_genes_set.intersection(filtered_marker_genes_set)
        
        # Add common genes to the list if not already present
        for gene in common_genes:
            if gene not in common_genes_by_celltype[cell_type]:
                common_genes_by_celltype[cell_type].append(gene)

common_genes_by_celltype

In [ ]:
#sc.pl.dotplot(adata, marker_genes_dict, groupby='leiden')
sc.pl.dotplot(adata, common_genes_by_celltype, groupby='leiden')
#sc.pl.dotplot(adata[adata.obs['sample'] == 'rgzj1'], common_genes_by_celltype, groupby='leiden')
#sc.pl.dotplot(adata[adata.obs['sample'] == 'rgzj2'], common_genes_by_celltype, groupby='leiden')
#sc.pl.dotplot(adata[adata.obs['sample'] == 'rgzj3'], common_genes_by_celltype, groupby='leiden')
#sc.pl.dotplot(adata[adata.obs['sample'] == 'rgzj4'], common_genes_by_celltype, groupby='leiden')
#sc.pl.dotplot(adata[adata.obs['sample'] == 'rgzj5'], common_genes_by_celltype, groupby='leiden')

## Excel Tables

In [ ]:
cluster_gene_table = sc.get.rank_genes_groups_df(adata, group= None)
df = pd.pivot_table(data=cluster_gene_table, index=['names'], columns=['group'])
#df

In [ ]:
with pd.ExcelWriter("AgingMuscle_ClusterMarkers.xlsx", engine='openpyxl') as writer:
    for clustername in df.columns.levels[1]:
        df.xs(clustername, level="group", axis=1).to_excel(writer, sheet_name=f"Cluster_{clustername}_vs. rest")

# Celltype Annotation

In [ ]:
target_gene = "Myh4"  # Replace with the actual gene name you're looking for

# Check if the gene is in the AnnData object's variable (gene) names
if target_gene in adata.var_names:
    print(f"{target_gene} is present in the AnnData object.")
else:
    print(f"{target_gene} is not present in the AnnData object.")


In [ ]:
full_annotation = {
    '0': 'Myonuclei',
    '1': 'Myonuclei',
    '2': 'FAPs',
    '3': 'Myonuclei (Myh1+)',
    '4': 'Myonuclei (differentiating myocytes?)',
    '5': 'Macrophages',
    '6': 'Myonuclei (myotendinous junction nuclei)',
    '7': 'Myonuclei (Myh4+?)',
    '8': 'Muscle stem cell',
    '9': 'Endothelial cells',
    '10': 'Fibroblasts',
    '11': 'Myonuclei (neuromuscular junction nuclei)',
    '12': 'Immune cells?',
    '13': 'Tenocytes',
    '14': 'Lymphatic vessels'
}

rough_annotation = {
    '0': 'Myonuclei',
    '1': 'Myonuclei',
    '2': 'Mono-nucleated cells',
    '3': 'Myonuclei',
    '4': 'Myonuclei',
    '5': 'Mono-nucleated cells',
    '6': 'Myonuclei',
    '7': 'Myonuclei',
    '8': 'Mono-nucleated cells',
    '9': 'Mono-nucleated cells',
    '10': 'Mono-nucleated cells',
    '11': 'Myonuclei',
    '12': 'Mono-nucleated cells',
    '13': 'Mono-nucleated cells',
    '14': 'Mono-nucleated cells'
}

mixed_annotation  = {
    '0': 'Myonuclei',
    '1': 'Myonuclei',
    '2': 'FAPs',
    '3': 'Myonuclei',
    '4': 'Myonuclei',
    '5': 'Macrophages',
    '6': 'Myonuclei',
    '7': 'Myonuclei',
    '8': 'Muscle stem cell',
    '9': 'Endothelial cells',
    '10': 'Fibroblasts',
    '11': 'Myonuclei',
    '12': 'Immune cells?',
    '13': 'Tenocytes',
    '14': 'Lymphatic vessels'
}

sub_annotation = {
    '0': 'Myonuclei',
    '1': 'Myonuclei',
    '2': 'FAPs',
    '3': 'Myh1+',
    '4': 'Differentiating myocytes?',
    '5': 'Macrophages',
    '6': 'Myotendinous junction nuclei',
    '7': 'Myh4+?',
    '8': 'Muscle stem cell',
    '9': 'Endothelial cells',
    '10': 'Fibroblasts?',
    '11': 'Neuromuscular junction nuclei',
    '12': 'Immune cells?',
    '13': 'Tenocytes',
    '14': 'Lymphatic vessels?'
}

# Assign cell type labels based on the leiden clustering results
adata.obs['celltype'] = [full_annotation[i] for i in adata.obs['leiden']]
adata.obs['subtype'] = [sub_annotation[i] for i in adata.obs['leiden']]
adata.obs['supertype'] = [rough_annotation[i] for i in adata.obs['leiden']]
adata.obs['mixed_celltype'] = [mixed_annotation[i] for i in adata.obs['leiden']]

# Write annotated output files

In [ ]:
adata_myo = adata[adata.obs['supertype']=='Myonuclei']
adata_ec = adata[adata.obs['mixed_celltype']=='Endothelial cells']
adata_neuro = adata[adata.obs['subtype']=='Neuromuscular junction nuclei']
adata_imm = adata[adata.obs['mixed_celltype']=='Macrophages']

outfile = '/data/bonn-epyc/projects/manuela/aging/aging_muscle/annotated-muscle-soupxed2.h5ad'
adata.write(outfile)

outfile2 = '/data/bonn-epyc/projects/manuela/aging/aging_muscle/annotated-myo_muscle-soupxed.h5ad'
adata_myo.write(outfile2)

outfile3 = '/data/bonn-epyc/projects/manuela/aging/aging_muscle/annotated-ec_muscle-soupxed.h5ad'
adata_ec.write(outfile3)

outfile4 = '/data/bonn-epyc/projects/manuela/aging/aging_muscle/annotated-neuro_muscle-soupxed.h5ad'
adata_neuro.write(outfile4)

outfile5 = '/data/bonn-epyc/projects/manuela/aging/aging_muscle/annotated-imm_muscle-soupxed.h5ad'
adata_neuro.write(outfile5)

# Visualizations

## Celltype composition

In [ ]:
tmp = pd.crosstab(adata.obs['subtype'],adata.obs['sample'], normalize='columns').T.plot(kind='bar', stacked=True)
tmp.legend(title='subtype', bbox_to_anchor=(1.9, 1),loc='upper right')

## Dotplot

In [ ]:
sc.pl.dotplot(adata, marker_genes_dict, groupby='celltype')
sc.pl.dotplot(adata, marker_genes_dict, groupby='subtype')
sc.pl.dotplot(adata, marker_genes_dict, groupby='supertype')

In [ ]:
sc.pl.umap(adata_myo, color=['subtype'] #,legend_loc='on data', frameon=False
              )
sc.pl.umap(adata_ec, color=['celltype'])
sc.pl.umap(adata_neuro, color=['subtype'])

In [ ]:
#adata.obs.clusters = adata.obs.clusters.astype('category')
with rc_context({'figure.figsize': (8, 8)}):
    sc.pl.umap(adata, color=['leiden'])
    sc.pl.umap(adata, color=['celltype'] #,legend_loc='on data', frameon=False
              )
    sc.pl.umap(adata, color=['subtype'])
    sc.pl.umap(adata, color=['supertype'])
    sc.pl.umap(adata, color=['mixed_celltype'])

## Celltype expansion across samples

In [ ]:
# Calculate cell type proportions in each sample
ct_sample_counts = pd.crosstab(adata.obs['subtype'], adata.obs['sample'])
#ct_sample_counts = pd.crosstab(adata.obs['sample'], adata.obs['subtype'])
ct_sample_proportions = ct_sample_counts.div(ct_sample_counts.sum(axis=1), axis=0)

# Create a heatmap of cell type proportions
plt.figure(figsize=(10, 10))
sns.heatmap(
    ct_sample_proportions,
    cmap='YlGnBu',  # Choose your preferred colormap
    annot=True, fmt=".2f",
    xticklabels=True,
    yticklabels=True,
    cbar_kws={'label': 'Cell Type Proportion'},
)
plt.title('Subtype Proportions in Samples')
plt.xlabel('Subtype')
plt.ylabel('Sample')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate cell type proportions in each sample
#ct_sample_counts = pd.crosstab(adata.obs['subtype'], adata.obs['sample'])
ct_sample_counts = pd.crosstab(adata.obs['sample'], adata.obs['subtype'])
ct_sample_proportions = ct_sample_counts.div(ct_sample_counts.sum(axis=1), axis=0)

# Create a heatmap of cell type proportions
plt.figure(figsize=(10, 10))
sns.heatmap(
    ct_sample_proportions,
    cmap='YlGnBu',  # Choose your preferred colormap
    annot=True, fmt=".2f",
    xticklabels=True,
    yticklabels=True,
    cbar_kws={'label': 'Cell Type Proportion'},
)
plt.title('Sample Proportions of Subtypes')
plt.xlabel('Subtype')
plt.ylabel('Sample')
plt.tight_layout()
plt.show()

## Lymphatic Marker Gene Plots

In [ ]:
# Assuming you have identified the optimal marker gene for lymphatic vessels
lymphatic_marker_gene= ['Pdpn', 'Lyve1', 'Prox1', 
                        'Flt4'
                       ]
#LYVE1

# Visualize expression of the lymphatic marker gene per Leiden cluster
sc.pl.dotplot(
    adata,
    var_names=lymphatic_marker_gene,  # List containing the lymphatic marker gene
    groupby='leiden',  # Group by the Leiden clusters
    use_raw=False,  # Use preprocessed data or raw data
    standard_scale='var',  # Standardize the expression for the gene
    figsize=(5, 6),  # Adjust the figure size
)

## DMD canonical cell marker plots

In [ ]:
#set(adata.obs.mixed_celltype)

In [ ]:
# canonical cell markers (FAP -> Dcn; ECs -> Cdh5; muscle stem cell-> Pax7; smooth muscle cells -> Acta2; myoblasts-> Myh4; inflammatory cells -> Ptprc)
marker_genes = ['Dcn', 'Cdh5', 'Pax7', 'Acta2', 'Myh4', 'Ptprc']

# Visualize expression of the lymphatic marker gene per Leiden cluster
sc.pl.dotplot(adata, var_names= marker_genes,  groupby='sample',  use_raw=False, standard_scale='var', figsize=(5, 6))
sc.pl.dotplot(adata, var_names= marker_genes,  groupby='mixed_celltype',  use_raw=False, standard_scale='var', figsize=(5, 6))


sc.pl.violin(adata, marker_genes, jitter =0.4, multi_panel= True, groupby = 'sample', rotation = 90)
sc.pl.violin(adata, marker_genes, jitter =0.4, multi_panel= True, groupby = 'mixed_celltype', rotation = 90)

## DMD fibroblast marker plots

In [ ]:
# fibroblast markers as Pdgfra, Decorin (Dcn), Vimentin (Vim) and S100a4
fibroblast_marker = ['Pdgfra', 'Dcn', 'Vim', 'S100a4']

# Visualize expression of the lymphatic marker gene per Leiden cluster
sc.pl.dotplot(adata, var_names= fibroblast_marker,  groupby='sample',  use_raw=False, standard_scale='var', figsize=(5, 6))
sc.pl.dotplot(adata, var_names= fibroblast_marker,  groupby='mixed_celltype',  use_raw=False, standard_scale='var', figsize=(5, 6))


sc.pl.violin(adata, fibroblast_marker, jitter =0.4, multi_panel= True, groupby = 'sample', rotation = 90)
sc.pl.violin(adata, fibroblast_marker, jitter =0.4, multi_panel= True, groupby = 'mixed_celltype', rotation = 90)

## DMD FAPs marker plots

In [ ]:
# fap markers as Pdgfra, Decorin (Dcn), Vimentin (Vim) and S100a4
fap_marker = ['Col8a1', 'Fabp4', 'Comp']

# Visualize expression of the lymphatic marker gene per Leiden cluster
sc.pl.dotplot(adata, var_names= fap_marker,  groupby='sample',  use_raw=False, standard_scale='var', figsize=(5, 6))
sc.pl.dotplot(adata, var_names= fap_marker,  groupby='mixed_celltype',  use_raw=False, standard_scale='var', figsize=(5, 6))


sc.pl.violin(adata, fap_marker, jitter =0.4, multi_panel= True, groupby = 'sample', rotation = 90)
sc.pl.violin(adata, fap_marker, jitter =0.4, multi_panel= True, groupby = 'mixed_celltype', rotation = 90)

# Counts per Cluster/ per Sample

In [ ]:
#Per cluster
counts = adata.obs['leiden'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
counts.plot(kind="bar", ax=ax)

# Add labels and title
plt.xlabel('Cluster Label')
plt.ylabel('Count')
plt.title('Cell Counts per Cluster')

# Show the plot
plt.show()

#Per Sample
counts = adata.obs['sample'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
counts.plot(kind="bar", ax=ax)

# Add labels and title
plt.xlabel('Cluster Label')
plt.ylabel('Count')
plt.title('Cell Counts per Sample')

# Show the plot
plt.show()

# create counts_per_sample dataframe
counts_per_sample = adata.obs.groupby(['sample', 'leiden']).size().unstack(fill_value=0)

# plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(counts_per_sample, cmap="YlGnBu")
plt.title("Cell counts per sample per cluster")
plt.show()

In [ ]:
# Calculate cell type proportions in each sample
ct_sample_counts = pd.crosstab(adata.obs['mixed_celltype'], adata.obs['sample'])
ct_sample_proportions = ct_sample_counts.div(ct_sample_counts.sum(axis=1), axis=0)

# Create a heatmap of cell type proportions
plt.figure(figsize=(10, 10))
sns.heatmap(
    ct_sample_proportions,
    cmap='YlGnBu',  # Choose your preferred colormap
    annot=True, fmt=".2f",
    xticklabels=True,
    yticklabels=True,
    cbar_kws={'label': 'Cell Type Proportion'},
)
plt.title('Subtype Proportions in Samples')
plt.ylabel('Celltype')
plt.xlabel('Sample')
plt.tight_layout()
plt.show()

# Downstream analyses

In [ ]:
adata_myo.uns['log1p']['base'] = None
adata_ec.uns['log1p']['base'] = None
adata_neuro.uns['log1p']['base'] = None
adata_imm.uns['log1p']['base'] = None

## Differential Expression Analysis (Ref. rgzj1)

### Immune cells

In [ ]:
# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(adata_imm, groupby='sample', reference='ctrl', method='t-test')

# Now plot the results
sc.pl.rank_genes_groups(adata_imm, n_genes=10)
sc.pl.rank_genes_groups_dotplot(adata_imm, n_genes=10)

for group in adata_imm.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = adata_imm.uns['rank_genes_groups']['names'][group][:20]
    sc.pl.violin(adata_imm, keys=top_genes, groupby='sample')
    
sc.get.rank_genes_groups_df(adata_imm, group = None)

In [ ]:
p = 0.05
log2fc = 0 

deg_imm_rgzj2 = sc.get.rank_genes_groups_df(adata_imm, group = 'rgzj2')

# significant genes -> filtered by pval and log2fc
deg_imm_sig_rgzj2 = deg_imm_rgzj2 [(deg_imm_rgzj2 ['pvals_adj'] < p) & (deg_imm_rgzj2 ['logfoldchanges'].abs() > log2fc)]
deg_imm_sig_up_rgzj2 = deg_imm_rgzj2 [(deg_imm_rgzj2 ['pvals_adj'] < p) & (deg_imm_rgzj2 ['logfoldchanges'] > 0 )]
deg_imm_sig_dwn_rgzj2 = deg_imm_rgzj2 [(deg_imm_rgzj2 ['pvals_adj'] < p) & (deg_imm_rgzj2 ['logfoldchanges'] < 0)]

print(deg_imm_rgzj2.shape)
print(deg_imm_sig_rgzj2.shape)
print(deg_imm_sig_up_rgzj2.shape)
print(deg_imm_sig_dwn_rgzj2.shape)

deg_imm_sig_rgzj2.to_csv('./tables/degs_imm_ctrl_aging.csv', index=False) 

#sc.pl.DotPlot(adata_imm, use_raw=True, var_names =deg_imm_sig_rgzj2.names, groupby='sample', standard_scale="var", title=f"imm DEGs ctrl vs. aging", cmap='BuGn', figsize=(300, 5)).savefig('./dotplot.pdf' ,format='pdf')
GeneExpression.volcano(deg_imm_rgzj2 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.05, 0.05), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')

In [ ]:
#deg_imm_sig_rgzj2['sample'].replace(value_mapping, inplace=True)
deg_imm_sig_rgzj2

### Myonuclei

In [ ]:
# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(adata_myo, groupby='sample', reference='rgzj1', method='t-test')

# Now plot the results
sc.pl.rank_genes_groups(adata_myo, n_genes=20)
sc.pl.rank_genes_groups_dotplot(adata_myo, n_genes=20)

for group in adata_myo.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = adata_myo.uns['rank_genes_groups']['names'][group][:20]
    sc.pl.violin(adata_myo, keys=top_genes, groupby='sample')
    
sc.get.rank_genes_groups_df(adata_myo, group = None)

In [ ]:
p = 0.05
log2fc = 0 

deg_myo_rgzj2 = sc.get.rank_genes_groups_df(adata_myo, group = 'rgzj2')

# significant genes -> filtered by pval and log2fc
deg_myo_sig_rgzj2 = deg_myo_rgzj2 [(deg_myo_rgzj2 ['pvals_adj'] < p) & (deg_myo_rgzj2 ['logfoldchanges'].abs() > log2fc)]
deg_myo_sig_up_rgzj2 = deg_myo_rgzj2 [(deg_myo_rgzj2 ['pvals_adj'] < p) & (deg_myo_rgzj2 ['logfoldchanges'] > 0 )]
deg_myo_sig_dwn_rgzj2 = deg_myo_rgzj2 [(deg_myo_rgzj2 ['pvals_adj'] < p) & (deg_myo_rgzj2 ['logfoldchanges'] < 0)]

print(deg_myo_rgzj2.shape)
print(deg_myo_sig_rgzj2.shape)
print(deg_myo_sig_up_rgzj2.shape)
print(deg_myo_sig_dwn_rgzj2.shape)

deg_myo_sig_rgzj2.to_csv('./tables/degs_myo_ctrl_aging.csv', index=False) 

#sc.pl.DotPlot(adata_myo, use_raw=True, var_names =deg_myo_sig_rgzj2.names, groupby='sample', standard_scale="var", title=f"Myo DEGs ctrl vs. aging", cmap='BuGn', figsize=(300, 5)).savefig('./dotplot.pdf' ,format='pdf')
GeneExpression.volcano(deg_myo_rgzj2 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.05, 0.05), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')

In [ ]:
# Create a bar plot for log-fold changes
plt.figure(figsize=(400, 5))  # Adjust the figure size as needed
# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors = ['green' if x >= 0 else 'red' for x in  deg_myo_sig_rgzj2['logfoldchanges']]
plt.bar(deg_myo_sig_rgzj2['names'],  deg_myo_sig_rgzj2['logfoldchanges'], color=colors)
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)  # Add a legend plt.legend()
    
# Save or display the plot (customize the path as needed)
#plt.savefig(f'logfoldchanges_group_{group}.png')
plt.show() 

### Endothelial Cells

In [ ]:
# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(adata_ec, groupby='sample', reference='rgzj1', method='t-test')

# Now plot the results
sc.pl.rank_genes_groups(adata_ec, n_genes=20)
sc.pl.rank_genes_groups_dotplot(adata_ec, n_genes=20)

for group in adata_ec.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = adata_ec.uns['rank_genes_groups']['names'][group][:20]
    sc.pl.violin(adata_ec, keys=top_genes, groupby='sample')
    
sc.get.rank_genes_groups_df(adata_ec, group = None)

In [ ]:
#p = 0.05
p = 0.1
log2fc = 0 

deg_ec_rgzj2 = sc.get.rank_genes_groups_df(adata_ec, group = 'rgzj2')

# significant genes -> filtered by pval and log2fc
deg_ec_sig_rgzj2 = deg_ec_rgzj2 [(deg_ec_rgzj2 ['pvals_adj'] < p) & (deg_ec_rgzj2 ['logfoldchanges'].abs() > log2fc)]
deg_ec_sig_up_rgzj2 = deg_ec_rgzj2 [(deg_ec_rgzj2 ['pvals_adj'] < p) & (deg_ec_rgzj2 ['logfoldchanges'] > 0 )]
deg_ec_sig_dwn_rgzj2 = deg_ec_rgzj2 [(deg_ec_rgzj2 ['pvals_adj'] < p) & (deg_ec_rgzj2 ['logfoldchanges'] < 0)]

print(deg_ec_rgzj2.shape)
print(deg_ec_sig_rgzj2.shape)
print(deg_ec_sig_up_rgzj2.shape)
print(deg_ec_sig_dwn_rgzj2.shape)
deg_ec_rgzj2#[deg_ec_rgzj2['pvals_adj']!=1.0]
deg_ec_sig_rgzj2.to_csv('./tables/degs_ec_ctrl_aging.csv', index=False) 

sc.pl.DotPlot(adata_ec, use_raw=True, var_names = deg_ec_sig_rgzj2.names, groupby='sample', standard_scale="var", title=f"EC DEGs ctrl vs. aging", cmap='BuGn', figsize=(5, 3)).savefig('./dotplot.pdf' ,format='pdf')
#GeneExpression.volcano(deg_ec_rgzj2 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.1, 0.1), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')

In [ ]:
# Create a bar plot for log-fold changes
plt.figure(figsize=(5, 5))  # Adjust the figure size as needed
# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors = ['green' if x >= 0 else 'red' for x in  deg_ec_sig_rgzj2['logfoldchanges']]
plt.bar(deg_ec_sig_rgzj2['names'],  deg_ec_sig_rgzj2['logfoldchanges'], color=colors)
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)  # Add a legend plt.legend()
    
# Save or display the plot (customize the path as needed)
#plt.savefig(f'logfoldchanges_group_{group}.png')
plt.show() 

### Neuromuscular Junction nuclei


In [ ]:
# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(adata_neuro, groupby='sample', reference='rgzj1', method='t-test')

# Now plot the results
sc.pl.rank_genes_groups(adata_neuro, n_genes=20)
sc.pl.rank_genes_groups_dotplot(adata_neuro, n_genes=20)

for group in adata_neuro.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = adata_neuro.uns['rank_genes_groups']['names'][group][:20]
    sc.pl.violin(adata_neuro, keys=top_genes, groupby='sample')
    
sc.get.rank_genes_groups_df(adata_neuro, group = None)

In [ ]:
#p = 0.05
p = 0.1
log2fc = 0 

deg_nmj_rgzj2 = sc.get.rank_genes_groups_df(adata_neuro, group = 'rgzj2')

# significant genes -> filtered by pval and log2fc
deg_nmj_sig_rgzj2 = deg_nmj_rgzj2 [(deg_nmj_rgzj2 ['pvals_adj'] < p) & (deg_nmj_rgzj2 ['logfoldchanges'].abs() > log2fc)]
deg_nmj_sig_up_rgzj2 = deg_nmj_rgzj2 [(deg_nmj_rgzj2 ['pvals_adj'] < p) & (deg_nmj_rgzj2 ['logfoldchanges'] > 0 )]
deg_nmj_sig_dwn_rgzj2 = deg_nmj_rgzj2 [(deg_nmj_rgzj2 ['pvals_adj'] < p) & (deg_nmj_rgzj2 ['logfoldchanges'] < 0)]

print(deg_nmj_rgzj2.shape)
print(deg_nmj_sig_rgzj2.shape)
print(deg_nmj_sig_up_rgzj2.shape)
print(deg_nmj_sig_dwn_rgzj2.shape)
deg_nmj_sig_rgzj2

deg_nmj_sig_rgzj2.to_csv('./tables/degs_nmj_ctrl_aging.csv', index=False) 
sc.pl.DotPlot(adata_neuro, use_raw=True, var_names = deg_nmj_sig_rgzj2.names, groupby='sample', standard_scale="var", title=f"NMJ DEGs ctrl vs. aging", cmap='BuGn', figsize=(5, 3)).savefig('./dotplot.pdf' ,format='pdf')

GeneExpression.volcano(deg_nmj_rgzj2 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.1, 0.1), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')

In [ ]:
# Create a bar plot for log-fold changes
plt.figure(figsize=(5, 5))  # Adjust the figure size as needed
# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors = ['green' if x >= 0 else 'red' for x in  deg_nmj_sig_rgzj2['logfoldchanges']]
plt.bar(deg_nmj_sig_rgzj2['names'],  deg_nmj_sig_rgzj2['logfoldchanges'], color=colors)
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)  # Add a legend plt.legend()
    
# Save or display the plot (customize the path as needed)
#plt.savefig(f'logfoldchanges_group_{group}.png')
plt.show() 

## Differential Expression Analysis (Ref. rgzj2)

### Myonuclei

In [ ]:
# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(adata_myo, groupby='sample', reference='rgzj2', method='t-test')

# Now plot the results
sc.pl.rank_genes_groups(adata_myo, n_genes=20)
sc.pl.rank_genes_groups_dotplot(adata_myo, n_genes=20)

for group in adata_myo.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = adata_myo.uns['rank_genes_groups']['names'][group][:20]
    sc.pl.violin(adata_myo, keys=top_genes, groupby='sample')
    
sc.get.rank_genes_groups_df(adata_myo, group = None)

In [ ]:
p = 0.05
log2fc = 0 

deg_myo_rgzj3 = sc.get.rank_genes_groups_df(adata_myo, group = 'rgzj3')
deg_myo_rgzj4 = sc.get.rank_genes_groups_df(adata_myo, group = 'rgzj4')
deg_myo_rgzj5 = sc.get.rank_genes_groups_df(adata_myo, group = 'rgzj5')

# significant genes -> filtered by pval and log2fc
deg_myo_sig_rgzj3 = deg_myo_rgzj3[(deg_myo_rgzj3['pvals_adj'] < p) & (deg_myo_rgzj3['logfoldchanges'].abs() > log2fc)]
deg_myo_sig_rgzj4 = deg_myo_rgzj4[(deg_myo_rgzj4['pvals_adj'] < p) & (deg_myo_rgzj4['logfoldchanges'].abs() > log2fc)]
deg_myo_sig_rgzj5 = deg_myo_rgzj5[(deg_myo_rgzj5['pvals_adj'] < p) & (deg_myo_rgzj5['logfoldchanges'].abs() > log2fc)]

deg_myo_sig_up_rgzj3 = deg_myo_rgzj3[(deg_myo_rgzj3['pvals_adj'] < p) & (deg_myo_rgzj3['logfoldchanges'] > 0 )]
deg_myo_sig_up_rgzj4 = deg_myo_rgzj4[(deg_myo_rgzj4['pvals_adj'] < p) & (deg_myo_rgzj4['logfoldchanges'] > 0 )]
deg_myo_sig_up_rgzj5 = deg_myo_rgzj5[(deg_myo_rgzj5['pvals_adj'] < p) & (deg_myo_rgzj5['logfoldchanges'] > 0 )]

deg_myo_sig_dwn_rgzj3 = deg_myo_rgzj3[(deg_myo_rgzj3['pvals_adj'] < p) & (deg_myo_rgzj3['logfoldchanges'] < 0)]
deg_myo_sig_dwn_rgzj4 = deg_myo_rgzj4[(deg_myo_rgzj4['pvals_adj'] < p) & (deg_myo_rgzj4['logfoldchanges'] < 0)]
deg_myo_sig_dwn_rgzj5 = deg_myo_rgzj5[(deg_myo_rgzj5['pvals_adj'] < p) & (deg_myo_rgzj5['logfoldchanges'] < 0)]

print('rgzj3: ',deg_myo_rgzj3.shape)
print(deg_myo_sig_rgzj3.shape)
print(deg_myo_sig_up_rgzj3.shape)
print(deg_myo_sig_dwn_rgzj3.shape)

print('rgzj4: ',deg_myo_rgzj4.shape)
print(deg_myo_sig_rgzj4.shape)
print(deg_myo_sig_up_rgzj4.shape)
print(deg_myo_sig_dwn_rgzj4.shape)

print('rgzj5: ',deg_myo_rgzj5.shape)
print(deg_myo_sig_rgzj5.shape)
print(deg_myo_sig_up_rgzj5.shape)
print(deg_myo_sig_dwn_rgzj5.shape)

deg_myo_sig_rgzj3.to_csv('./tables/degs_myo_aging_sAct.csv', index=False) 
deg_myo_sig_rgzj4.to_csv('./tables/degs_myo_aging_DR.csv', index=False) 
deg_myo_sig_rgzj5.to_csv('./tables/degs_myo_aging_comb.csv', index=False) 

GeneExpression.volcano(deg_myo_rgzj3 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.05, 0.05), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')
GeneExpression.volcano(deg_myo_rgzj4 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.05, 0.05), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')
GeneExpression.volcano(deg_myo_rgzj5 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.05, 0.05), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')

### Endothelial Cells

In [ ]:
# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(adata_ec, groupby='sample', reference='rgzj2', method='t-test')

# Now plot the results
sc.pl.rank_genes_groups(adata_ec, n_genes=20)
sc.pl.rank_genes_groups_dotplot(adata_ec, n_genes=20)

for group in adata_ec.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = adata_ec.uns['rank_genes_groups']['names'][group][:20]
    sc.pl.violin(adata_ec, keys=top_genes, groupby='sample')
    
sc.get.rank_genes_groups_df(adata_ec, group = None)

In [ ]:
#p = 0.05
p = 0.1
log2fc = 0 

deg_ec_rgzj3 = sc.get.rank_genes_groups_df(adata_ec, group = 'rgzj3')
deg_ec_rgzj4 = sc.get.rank_genes_groups_df(adata_ec, group = 'rgzj4')
deg_ec_rgzj5 = sc.get.rank_genes_groups_df(adata_ec, group = 'rgzj5')

# significant genes -> filtered by pval and log2fc
deg_ec_sig_rgzj3 = deg_ec_rgzj3[(deg_ec_rgzj3['pvals_adj'] < p) & (deg_ec_rgzj3['logfoldchanges'].abs() > log2fc)]
deg_ec_sig_rgzj4 = deg_ec_rgzj4[(deg_ec_rgzj4['pvals_adj'] < p) & (deg_ec_rgzj4['logfoldchanges'].abs() > log2fc)]
deg_ec_sig_rgzj5 = deg_ec_rgzj5[(deg_ec_rgzj5['pvals_adj'] < p) & (deg_ec_rgzj5['logfoldchanges'].abs() > log2fc)]

deg_ec_sig_up_rgzj3 = deg_ec_rgzj3[(deg_ec_rgzj3['pvals_adj'] < p) & (deg_ec_rgzj3['logfoldchanges'] > 0 )]
deg_ec_sig_up_rgzj4 = deg_ec_rgzj4[(deg_ec_rgzj4['pvals_adj'] < p) & (deg_ec_rgzj4['logfoldchanges'] > 0 )]
deg_ec_sig_up_rgzj5 = deg_ec_rgzj5[(deg_ec_rgzj5['pvals_adj'] < p) & (deg_ec_rgzj5['logfoldchanges'] > 0 )]

deg_ec_sig_dwn_rgzj3 = deg_ec_rgzj3[(deg_ec_rgzj3['pvals_adj'] < p) & (deg_ec_rgzj3['logfoldchanges'] < 0)]
deg_ec_sig_dwn_rgzj4 = deg_ec_rgzj4[(deg_ec_rgzj4['pvals_adj'] < p) & (deg_ec_rgzj4['logfoldchanges'] < 0)]
deg_ec_sig_dwn_rgzj5 = deg_ec_rgzj5[(deg_ec_rgzj5['pvals_adj'] < p) & (deg_ec_rgzj5['logfoldchanges'] < 0)]

print('rgzj3: ', deg_ec_rgzj3.shape)
print(deg_ec_sig_rgzj3.shape)
print(deg_ec_sig_up_rgzj3.shape)
print(deg_ec_sig_dwn_rgzj3.shape)
print(deg_ec_sig_rgzj3)

print('rgzj4: ',deg_ec_rgzj4.shape)
print(deg_ec_sig_rgzj4.shape)
print(deg_ec_sig_up_rgzj4.shape)
print(deg_ec_sig_dwn_rgzj4.shape)
print(deg_ec_sig_rgzj4)

print('rgzj5: ',deg_ec_rgzj5.shape)
print(deg_ec_sig_rgzj5.shape)
print(deg_ec_sig_up_rgzj5.shape)
print(deg_ec_sig_dwn_rgzj5.shape)
print(deg_ec_sig_rgzj5)

deg_ec_sig_rgzj3.to_csv('./tables/degs_ec_aging_sAct.csv', index=False) 
deg_ec_sig_rgzj4.to_csv('./tables/degs_ec_aging_DR.csv', index=False) 
deg_ec_sig_rgzj5.to_csv('./tables/degs_ec_aging_comb.csv', index=False) 


#GeneExpression.volcano(deg_ec_rgzj3 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.1, 0.1), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')
#GeneExpression.volcano(deg_ec_rgzj4 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.1, 0.1), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')
GeneExpression.volcano(deg_ec_rgzj5 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.1, 0.1), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')

### Neuromuscular Junction nuclei

In [ ]:
# Perform the differential expression analysis for the subset
sc.tl.rank_genes_groups(adata_neuro, groupby='sample', reference='rgzj2', method='t-test')

# Now plot the results
sc.pl.rank_genes_groups(adata_neuro, n_genes=20)
sc.pl.rank_genes_groups_dotplot(adata_neuro, n_genes=20)

for group in adata_neuro.uns['rank_genes_groups']['names'].dtype.names:
    top_genes = adata_neuro.uns['rank_genes_groups']['names'][group][:20]
    sc.pl.violin(adata_neuro, keys=top_genes, groupby='sample')
    
sc.get.rank_genes_groups_df(adata_neuro, group = None)

In [ ]:
#p = 0.05
p = 0.1
log2fc = 0 

deg_nmj_rgzj3 = sc.get.rank_genes_groups_df(adata_neuro, group = 'rgzj3')
deg_nmj_rgzj4 = sc.get.rank_genes_groups_df(adata_neuro, group = 'rgzj4')
deg_nmj_rgzj5 = sc.get.rank_genes_groups_df(adata_neuro, group = 'rgzj5')

# significant genes -> filtered by pval and log2fc
deg_nmj_sig_rgzj3 = deg_nmj_rgzj3[(deg_nmj_rgzj3['pvals_adj'] < p) & (deg_nmj_rgzj3['logfoldchanges'].abs() > log2fc)]
deg_nmj_sig_rgzj4 = deg_nmj_rgzj4[(deg_nmj_rgzj4['pvals_adj'] < p) & (deg_nmj_rgzj4['logfoldchanges'].abs() > log2fc)]
deg_nmj_sig_rgzj5 = deg_nmj_rgzj5[(deg_nmj_rgzj5['pvals_adj'] < p) & (deg_nmj_rgzj5['logfoldchanges'].abs() > log2fc)]

deg_nmj_sig_up_rgzj3 = deg_nmj_rgzj3[(deg_nmj_rgzj3['pvals_adj'] < p) & (deg_nmj_rgzj3['logfoldchanges'] > 0 )]
deg_nmj_sig_up_rgzj4 = deg_nmj_rgzj4[(deg_nmj_rgzj4['pvals_adj'] < p) & (deg_nmj_rgzj4['logfoldchanges'] > 0 )]
deg_nmj_sig_up_rgzj5 = deg_nmj_rgzj5[(deg_nmj_rgzj5['pvals_adj'] < p) & (deg_nmj_rgzj5['logfoldchanges'] > 0 )]

deg_nmj_sig_dwn_rgzj3 = deg_nmj_rgzj3[(deg_nmj_rgzj3['pvals_adj'] < p) & (deg_nmj_rgzj3['logfoldchanges'] < 0)]
deg_nmj_sig_dwn_rgzj4 = deg_nmj_rgzj4[(deg_nmj_rgzj4['pvals_adj'] < p) & (deg_nmj_rgzj4['logfoldchanges'] < 0)]
deg_nmj_sig_dwn_rgzj5 = deg_nmj_rgzj5[(deg_nmj_rgzj5['pvals_adj'] < p) & (deg_nmj_rgzj5['logfoldchanges'] < 0)]

print('rgzj3: ', deg_nmj_rgzj3.shape)
print(deg_nmj_sig_rgzj3.shape)
print(deg_nmj_sig_up_rgzj3.shape)
print(deg_nmj_sig_dwn_rgzj3.shape)

print('rgzj4: ', deg_nmj_rgzj4.shape)
print(deg_nmj_sig_rgzj4.shape)
print(deg_nmj_sig_up_rgzj4.shape)
print(deg_nmj_sig_dwn_rgzj4.shape)

print('rgzj5: ', deg_nmj_rgzj5.shape)
print(deg_nmj_sig_rgzj5.shape)
print(deg_nmj_sig_up_rgzj5.shape)
print(deg_nmj_sig_dwn_rgzj5.shape)

deg_nmj_sig_rgzj3.to_csv('./tables/degs_nmj_aging_sAct.csv', index=False) 
deg_nmj_sig_rgzj4.to_csv('./tables/degs_nmj_aging_DR.csv', index=False) 
deg_nmj_sig_rgzj5.to_csv('./tables/degs_nmj_aging_comb.csv', index=False) 

print('rgzj3: ',deg_nmj_sig_rgzj3)
print('rgzj4: ',deg_nmj_sig_rgzj4)
print('rgzj5: ',deg_nmj_sig_rgzj5)
#GeneExpression.volcano(deg_nmj_rgzj3 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.1, 0.1), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')
GeneExpression.volcano(deg_nmj_rgzj4 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.1, 0.1), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')
GeneExpression.volcano(deg_nmj_rgzj5 , lfc='logfoldchanges', pv='pvals_adj', lfc_thr = (0,0), pv_thr = (0.1, 0.1), color=('red', 'grey', 'blue'),gfont=8, dim=(10, 6), show=True, figname = 'hallo', axtickfontsize=8, axtickfontname='Arial', axlabelfontsize=10, axlabelfontname='Arial')

## Intersection 

### Myonuclei

In [ ]:
#print("\033[1m" + 'Regulated by aging: '+ "\033[0m" ,deg_myo_sig_rgzj2)
print("\033[1m" + 'Upregulated by aging: '+ "\033[0m" ,deg_myo_sig_up_rgzj2)
print("\033[1m" + 'Downregulated by aging: '+ "\033[0m",deg_myo_sig_dwn_rgzj2)

print("\033[1m" + 'Regulated by sAct: '+ "\033[0m" , deg_myo_sig_rgzj3)
#print("\033[1m" + 'Upregulated by sAct: '+ "\033[0m" , deg_myo_sig_up_rgzj3)
#print("\033[1m" + 'Downregulated by sAct: '+ "\033[0m" , deg_myo_sig_dwn_rgzj3)

print("\033[1m" + 'Regulated by DR: '+ "\033[0m" , deg_myo_sig_rgzj4)
#print("\033[1m" + 'Upregulated by DR: '+ "\033[0m" , deg_myo_sig_up_rgzj4)
#print("\033[1m" + 'Downregulated by DR: '+ "\033[0m" , deg_myo_sig_dwn_rgzj4)

print("\033[1m" + 'Regulated by combi: '+ "\033[0m" , deg_myo_sig_rgzj5)
#print("\033[1m" + 'Upregulated by combi: '+ "\033[0m" , deg_myo_sig_up_rgzj5)
#print("\033[1m" + 'Downregulated by combi: '+ "\033[0m" , deg_myo_sig_dwn_rgzj5)

In [ ]:
# Use merge on aging dwn - rescued by rgzj3 
result = pd.merge(deg_myo_sig_dwn_rgzj2, deg_myo_sig_rgzj3, on='names', suffixes=('_age', '_treat'))
degs_myo_agedwn_sAct = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]

degs_myo_agedwn_sAct.to_csv('./tables/degs_myo_agedwn_sAct.csv', index=False) 

In [ ]:
#result
# Create a bar plot for log-fold changes
plt.figure(figsize=(250, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_myo_agedwn_sAct['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_myo_agedwn_sAct['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_myo_agedwn_sAct['names'], degs_myo_agedwn_sAct['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_myo_agedwn_sAct['names'], degs_myo_agedwn_sAct['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)  # Add a legend 
plt.legend()
# Set margin
plt.subplots_adjust(bottom=0.4)
    
# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_myo_agedwn_sAct.png')
plt.show() 

In [ ]:
# Use merge on aging dwn - rescued by rgzj4
result = pd.merge(deg_myo_sig_dwn_rgzj2, deg_myo_sig_rgzj4, on='names', suffixes=('_age', '_treat'))
degs_myo_agedwn_DR = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]
#result

degs_myo_agedwn_DR.to_csv('./tables/degs_myo_agedwn_DR.csv', index=False) 
#result

In [ ]:
# Create a bar plot for log-fold changes
plt.figure(figsize=(250, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_myo_agedwn_DR['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_myo_agedwn_DR['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_myo_agedwn_DR['names'], degs_myo_agedwn_DR['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_myo_agedwn_DR['names'], degs_myo_agedwn_DR['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)  
plt.legend()

# Set margin
plt.subplots_adjust(bottom=0.4)
    
# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_myo_agedwn_DR.png')
plt.show() 

In [ ]:
# Use merge on aging dwn - rescued by rgzj5
result = pd.merge(deg_myo_sig_dwn_rgzj2, deg_myo_sig_rgzj5, on='names', suffixes=('_age', '_treat'))
degs_myo_agedwn_comb = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]

degs_myo_agedwn_comb.to_csv('./tables/degs_myo_agedwn_comb.csv', index=False) 

#result
degs_myo_agedwn_comb

In [ ]:
# Create a bar plot for log-fold changes
plt.figure(figsize=(250, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_myo_agedwn_comb['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_myo_agedwn_comb['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_myo_agedwn_comb['names'], degs_myo_agedwn_comb['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_myo_agedwn_comb['names'], degs_myo_agedwn_comb['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)
plt.legend()
# Set margin
plt.subplots_adjust(bottom=0.4)

# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_myo_agedwn_comb.png')
plt.show() 

In [ ]:
# Use merge on aging up - rescued by rgzj3 
result = pd.merge(deg_myo_sig_up_rgzj2, deg_myo_sig_rgzj3, on='names', suffixes=('_age', '_treat'))
# Assuming you have the result data frame from the previous step
degs_myo_ageup_sAct = result[result['logfoldchanges_age'] > result['logfoldchanges_treat']]

degs_myo_ageup_sAct.to_csv('./tables/degs_myo_ageup_sAct.csv', index=False) 
#result

In [ ]:
# Create a bar plot for log-fold changes
plt.figure(figsize=(250, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_myo_ageup_sAct['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_myo_ageup_sAct['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_myo_ageup_sAct['names'], degs_myo_ageup_sAct['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_myo_ageup_sAct['names'], degs_myo_ageup_sAct['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)  
plt.legend()
# Set margin
plt.subplots_adjust(bottom=0.4)
    
# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_myo_ageup_sAct.png')
plt.show() 

In [ ]:
# Use merge on aging up - rescued by rgzj4 
result = pd.merge(deg_myo_sig_up_rgzj2, deg_myo_sig_rgzj4, on='names', suffixes=('_age', '_treat'))
# Assuming you have the result data frame from the previous step
degs_myo_ageup_DR = result[result['logfoldchanges_age'] > result['logfoldchanges_treat']]

degs_myo_ageup_DR.to_csv('./tables/degs_myo_ageup_DR.csv', index=False) 
#result

In [ ]:
# Create a bar plot for log-fold changes
plt.figure(figsize=(270, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_myo_ageup_DR['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_myo_ageup_DR['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_myo_ageup_DR['names'], degs_myo_ageup_DR['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_myo_ageup_DR['names'], degs_myo_ageup_DR['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)
plt.legend()
# Set margin
plt.subplots_adjust(bottom=0.4)
    
# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_myo_ageup_DR.png')
plt.show() 

In [ ]:
# Use merge on aging up - rescued by rgzj5
result = pd.merge(deg_myo_sig_up_rgzj2, deg_myo_sig_rgzj5, on='names', suffixes=('_age', '_treat'))
# Assuming you have the result data frame from the previous step
degs_myo_ageup_comb = result[result['logfoldchanges_age'] > result['logfoldchanges_treat']]

degs_myo_ageup_comb.to_csv('./tables/degs_myo_ageup_comb.csv', index=False) 
#result

In [ ]:
# Create a bar plot for log-fold changes
plt.figure(figsize=(250, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
colors_age = ['green' if x >= 0 else 'red' for x in  degs_myo_ageup_comb['logfoldchanges_age']]
colors_treat = ['blue' if x >= 0 else 'orange' for x in  degs_myo_ageup_comb['logfoldchanges_treat']]

# Plot the bar for logfoldchanges_age
plt.bar(degs_myo_ageup_comb['names'], degs_myo_ageup_comb['logfoldchanges_age'], color=colors_age, width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_myo_ageup_comb['names'], degs_myo_ageup_comb['logfoldchanges_treat'], color=colors_treat, width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)
plt.legend()
# Set margin
plt.subplots_adjust(bottom=0.4)
    
# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_myo_ageup_comb.png')
plt.show() 

### ECs

In [ ]:
#deg_ec_sig_dwn_rgzj2
#deg_ec_sig_rgzj3
#deg_ec_sig_rgzj4
deg_ec_sig_rgzj5

In [ ]:
# Use merge on aging dwn - rescued by rgzj3 
result = pd.merge(deg_ec_sig_dwn_rgzj2, deg_ec_sig_rgzj3, on='names', suffixes=('_age', '_treat'))
degs_ec_agedwn_sAct = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]

degs_ec_agedwn_sAct.to_csv('./tables/degs_ec_agedwn_sAct.csv', index=False) 

#result
degs_ec_agedwn_sAct

In [ ]:
# Use merge on aging dwn - rescued by rgzj4
result = pd.merge(deg_ec_sig_dwn_rgzj2, deg_ec_sig_rgzj4, on='names', suffixes=('_age', '_treat'))
degs_ec_agedwn_DR = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]

degs_ec_agedwn_DR.to_csv('./tables/degs_ec_agedwn_DR.csv', index=False) 

#result
degs_ec_agedwn_DR

In [ ]:
# Use merge on aging dwn - rescued by rgzj5
result = pd.merge(deg_ec_sig_dwn_rgzj2, deg_ec_sig_rgzj5, on='names', suffixes=('_age', '_treat'))
degs_ec_agedwn_combi = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]

degs_ec_agedwn_combi.to_csv('./tables/degs_ec_agedwn_combi.csv', index=False) 

#result
degs_ec_agedwn_combi

In [ ]:
# Use merge on aging up - rescued by rgzj4 
result = pd.merge(deg_ec_sig_up_rgzj2, deg_ec_sig_rgzj4, on='names', suffixes=('_age', '_treat'))
# Assuming you have the result data frame from the previous step
degs_ec_ageup_DR = result[result['logfoldchanges_age'] > result['logfoldchanges_treat']]

degs_ec_ageup_DR.to_csv('./tables/degs_ec_ageup_DR.csv', index=False) 
#result
degs_ec_ageup_DR

In [ ]:
# Use merge on aging up - rescued by rgzj5
result = pd.merge(deg_ec_sig_up_rgzj2, deg_ec_sig_rgzj5, on='names', suffixes=('_age', '_treat'))
# Assuming you have the result data frame from the previous step
degs_ec_ageup_comb = result[result['logfoldchanges_age'] > result['logfoldchanges_treat']]

degs_ec_ageup_comb.to_csv('./tables/degs_ec_ageup_comb.csv', index=False) 
#result
degs_ec_ageup_comb

### NMJs

In [ ]:
# Use merge on aging dwn - rescued by rgzj3 
result = pd.merge(deg_nmj_sig_dwn_rgzj2, deg_nmj_sig_rgzj3, on='names', suffixes=('_age', '_treat'))
degs_nmj_agedwn_sAct = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]

degs_nmj_agedwn_sAct.to_csv('./tables/degs_nmj_agedwn_sAct.csv', index=False) 

result
#degs_nmj_agedwn_sAct

In [ ]:
# Use merge on aging dwn - rescued by rgzj4 
result = pd.merge(deg_nmj_sig_dwn_rgzj2, deg_nmj_sig_rgzj4, on='names', suffixes=('_age', '_treat'))
degs_nmj_agedwn_DR = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]

degs_nmj_agedwn_DR.to_csv('./tables/degs_nmj_agedwn_DR.csv', index=False) 

result
#degs_nmj_agedwn_DR

In [ ]:
# Use merge on aging dwn - rescued by rgzj5 
result = pd.merge(deg_nmj_sig_dwn_rgzj2, deg_nmj_sig_rgzj5, on='names', suffixes=('_age', '_treat'))
degs_nmj_agedwn_comb = result[result['logfoldchanges_age'] < result['logfoldchanges_treat']]

degs_nmj_agedwn_comb.to_csv('./tables/degs_nmj_agedwn_comb.csv', index=False) 

#result
degs_nmj_agedwn_comb

# Create a bar plot for log-fold changes
plt.figure(figsize=(5, 5))  # Adjust the figure size as needed

# Use 'green' for positive log-fold changes and 'red' for negative log-fold changes
#colors = ['green' if x >= 0 else 'red' for x in  degs_nmj_agedwn_comb['logfoldchanges_age']]
#plt.bar(degs_nmj_agedwn_comb['names'],  degs_nmj_agedwn_comb['logfoldchanges_age'], color=colors)

# Plot the bar for logfoldchanges_age
plt.bar(degs_nmj_agedwn_comb['names'], degs_nmj_agedwn_comb['logfoldchanges_age'], color='red', width=0.4, align='center', label='logfoldchanges_age')

# Plot the bar for logfoldchanges_treat next to the first bar
plt.bar(degs_nmj_agedwn_comb['names'], degs_nmj_agedwn_comb['logfoldchanges_treat'], color='green', width=0.4, align='edge', label='logfoldchanges_treat')
plt.title(f'Log-fold Changes for sig. DEGs ')
plt.xlabel('Genes')
plt.ylabel('Log-fold Change')
plt.xticks(rotation=90)  
plt.legend()
# Set margin
plt.subplots_adjust(bottom=0.4)
    
# Save or display the plot (customize the path as needed)
plt.savefig(f'logfoldchanges_nmj_agedwn_sAct.png')
plt.show() 

In [ ]:
# Use merge on aging up - rescued by rgzj3 
result = pd.merge(deg_nmj_sig_up_rgzj2, deg_nmj_sig_rgzj3, on='names', suffixes=('_age', '_treat'))
# Assuming you have the result data frame from the previous step
degs_nmj_ageup_sAct = result[result['logfoldchanges_age'] > result['logfoldchanges_treat']]

degs_nmj_ageup_sAct.to_csv('./tables/degs_nmj_ageup_sAct.csv', index=False) 
#result
degs_nmj_ageup_sAct

# Use merge on aging up - rescued by rgzj4 
result = pd.merge(deg_nmj_sig_up_rgzj2, deg_nmj_sig_rgzj4, on='names', suffixes=('_age', '_treat'))
# Assuming you have the result data frame from the previous step
degs_nmj_ageup_DR = result[result['logfoldchanges_age'] > result['logfoldchanges_treat']]

degs_nmj_ageup_DR.to_csv('./tables/degs_nmj_ageup_DR.csv', index=False) 
#result
degs_nmj_ageup_DR

In [ ]:
# Use merge on aging up - rescued by rgzj5 
result = pd.merge(deg_nmj_sig_up_rgzj2, deg_nmj_sig_rgzj5, on='names', suffixes=('_age', '_treat'))
# Assuming you have the result data frame from the previous step
degs_nmj_ageup_comb = result[result['logfoldchanges_age'] > result['logfoldchanges_treat']]

degs_nmj_ageup_comb.to_csv('./tables/degs_nmj_ageup_comb.csv', index=False) 
#result
degs_nmj_ageup_comb

## Enrichment - Tabula Muris

In [ ]:
gene_sets = gp.read_gmt('/data/bonn-epyc/projects/manuela/aging/aging_muscle/msigdb/m8.all.v2023.2.Mm.symbols.gmt')
# Filter the gene sets to only include those starting with 'TABULA_MURIS_SENIS'
gene_sets = {k: v for k, v in gene_sets.items() if k.startswith('TABULA_MURIS_SENIS')}
gene_sets = {k.split('_', maxsplit=3)[-1]: v for k, v in gene_sets.items()}
#

## Gene Set Enrichment Analysis (GSEA)

In [ ]:
from gseapy.scipalette import SciPalette
NbDr = SciPalette.create_colormap() 

### Immune cells

In [ ]:
value_mapping = {
    'SG18': 'ctrl',
    'SG20': 'age',
    'SG24': 'sAct',
    'SG26': 'DR',
    'SG28': 'combi',
    'sg18': 'ctrl',
    'sg20': 'age',
    'sg24': 'sAct',
    'sg26': 'DR',
    'sg28': 'combi',
    'rgzj1': 'ctrl',
    'rgzj2': 'age',
    'rgzj3': 'sAct',
    'rgzj4': 'DR',
    'rgzj5': 'combi'
}

adata_imm.obs['sample'].replace(value_mapping, inplace=True)

In [ ]:


sc.pl.dotplot(adata_imm, deg_imm_sig_rgzj2.names, groupby='sample', standard_scale="var", title = "Muscle ")

In [ ]:
import gseapy as gp
import seaborn as sns
import matplotlib.pyplot as plt
from gsea_api.molecular_signatures_db import GeneSets
from gsea_api.gsea import GSEApy

gene_set_file = GeneSets.from_gmt("/data/bonn-epyc/projects/manuela/aging_bulk/m8.all.v2023.2.Mm.symbols.gmt")
gsea = GSEApy()

# Perform enrichment analysis
enr_results = gp.enrichr(gene_list=deg_imm_sig_up_rgzj2.names,
                         gene_sets=gene_set_file,
                       #  description='MSigDB_mouse_collection',
                         cutoff=0.05)  # You can set cutoffs for p-value and q-value here

# Print results
print(enr_results.res2d)

# Visualize the results
sns.barplot(x='P-value', y='Term', data=enr_results.res2d.head(10))
plt.xlabel('P-value')
plt.ylabel('Gene Set')
plt.title('Top Enriched Gene Sets')
plt.show()

In [ ]:
from gsea_api.molecular_signatures_db import MolecularSignaturesDatabase

msigdb = MolecularSignaturesDatabase('msigdb')
print(msigdb.gene_sets)

m8_pathways = GeneSets.from_gmt('/data/bonn-epyc/projects/manuela/aging/aging_muscle/msigdb/m8.all.v2023.2.Mm.symbols.gmt')

print(len(m8_pathways.gene_sets))
#print(m8_pathways.gene_sets)

In [ ]:
m8_pathways.load('TABULA_MURIS_SENIS_SPLEEN_NK_CELL_AGEING', 'symbols')

In [ ]:
enr_up_m8 = gp.enrichr(deg_imm_sig_up_rgzj2.names,
                        gene_sets=['MSigDB_M8'],
                        outdir=None)
enr_dw_m8 = gp.enrichr(deg_imm_sig_dwn_rgzj2.names,
                        gene_sets=['TABULA_MURIS_SENIS_AORTA_AORTIC_ENDOTHELIAL_CELL_AGEING'],
                        outdir=None)

In [ ]:
enr_up = gp.enrichr(deg_imm_sig_up_rgzj2.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
enr_dw = gp.enrichr(deg_imm_sig_dwn_rgzj2.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(deg_imm_sig_up_rgzj2.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
enr_dw_go = gp.enrichr(deg_imm_sig_dwn_rgzj2.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)



# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Upregulated by aging in Immune cells", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Downregulated by aging in Immune cells", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Upregulated by aging in Immune cells", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Downregulated by aging in Immune cells", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_m8.res2d, figsize=(3, 5), top_term=20, title=f"M8 - Upregulated by aging in Immune cells", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_m8.res2d, figsize=(3, 5), top_term=20, title=f"M8 - Downregulated by aging in Immune cells", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

### Myonuclei

In [ ]:
enr_up = gp.enrichr(deg_myo_sig_up_rgzj2.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_dw = gp.enrichr(deg_myo_sig_dwn_rgzj2.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(deg_myo_sig_up_rgzj2.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
enr_dw_go = gp.enrichr(deg_myo_sig_dwn_rgzj2.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)



enr_up.results.to_csv('/data/bonn-epyc/projects/manuela/aging/aging_muscle/enrichment_tables/kegg_myo_upAging.csv')
enr_dw.results.to_csv('/data/bonn-epyc/projects/manuela/aging/aging_muscle/enrichment_tables/kegg_myo_dwAging.csv')
enr_up_go.results.to_csv('/data/bonn-epyc/projects/manuela/aging/aging_muscle/enrichment_tables/go_myo_upAging.csv')
enr_dw_go.results.to_csv('/data/bonn-epyc/projects/manuela/aging/aging_muscle/enrichment_tables/go_myo_dwAging.csv')


# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Upregulated by aging in Myonuclei", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Downregulated by aging in Myonuclei", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Upregulated by aging in Myonuclei", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Downregulated by aging in Myonuclei", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

In [ ]:
enr_up = gp.enrichr(degs_myo_ageup_sAct.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_dw = gp.enrichr(degs_myo_agedwn_sAct.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(degs_myo_ageup_sAct.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
enr_dw_go = gp.enrichr(degs_myo_agedwn_sAct.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Up in aging, rescued by sAct in Myonuclei", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Down in aging, rescued by sAct in Myonuclei", cmap= plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Up in aging, rescued by sAct in Myonuclei", cmap= plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Down in aging, rescued by sAct in Myonuclei", cmap= plt.cm.winter_r, size=5, cutoff = 0.5)

In [ ]:
enr_up = gp.enrichr(degs_myo_ageup_DR.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_dw = gp.enrichr(degs_myo_agedwn_DR.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(degs_myo_ageup_DR.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

enr_dw_go = gp.enrichr(degs_myo_agedwn_DR.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Up in aging, rescued by DR in Myonuclei", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Down in aging, rescued by DR in Myonuclei", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Up in aging, rescued by DR in Myonuclei", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Down in aging, rescued by DR in Myonuclei", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

In [ ]:
enr_up = gp.enrichr(degs_myo_ageup_comb.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_dw = gp.enrichr(degs_myo_agedwn_comb.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(degs_myo_ageup_comb.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

enr_dw_go = gp.enrichr(degs_myo_agedwn_comb.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Down in aging, rescued by combi in Myonuclei", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Up in aging, rescued by combi in Myonuclei", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Down in aging, rescued by combi in Myonuclei", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Up in aging, rescued by combi in Myonuclei", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

### Endothelial Cells

In [ ]:
#sc.pl.DotPlot(adata_ec, use_raw=True, var_names = deg_ec_sig_up_rgzj2.names, groupby='sample', standard_scale="var", title=f"Upregulated by aging", cmap='BuGn', figsize=(50, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
#sc.pl.DotPlot(adata_ec, use_raw=True, var_names = deg_ec_sig_dwn_rgzj2.names, groupby='sample', standard_scale="var", title=f" Downregulated by aging", cmap='BuGn', figsize=(50, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')

#enr_up = gp.enrichr(deg_ec_sig_up_rgzj2.names, gene_sets=['KEGG_2019_Mouse'], outdir=None)

enr_dw = gp.enrichr(deg_ec_sig_dwn_rgzj2.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

#enr_up_go = gp.enrichr(deg_ec_sig_up_rgzj2.names, gene_sets=['GO_Biological_Process_2021'], outdir=None)

enr_dw_go = gp.enrichr(deg_ec_sig_dwn_rgzj2.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Downregulated by aging in EC", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Downregulated by aging in EC", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

In [ ]:
enr_up = gp.enrichr(degs_ec_agedwn_sAct.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_dw = gp.enrichr(degs_ec_ageup_sAct.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(degs_ec_agedwn_sAct.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
enr_dw_go = gp.enrichr(degs_ec_ageup_sAct.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Up in aging, rescued by sAct in Myonuclei", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Down in aging, rescued by sAct in Myonuclei", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Up in aging, rescued by sAct in Myonuclei", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Down in aging, rescued by sAct in Myonuclei", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

In [ ]:
print("\033[1m" + 'Regulated by DR: '+ "\033[0m" , deg_ec_sig_rgzj4)
#print("\033[1m" + 'Upregulated by DR: '+ "\033[0m" , deg_ec_sig_up_rgzj4)
#print("\033[1m" + 'Downregulated by DR: '+ "\033[0m" , deg_ec_sig_dwn_rgzj4)


In [ ]:
print("\033[1m" + 'Regulated by combi: '+ "\033[0m" , deg_ec_sig_rgzj5)

### NMJs

In [ ]:
enr_up = gp.enrichr(deg_nmj_sig_up_rgzj2.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_dw = gp.enrichr(deg_nmj_sig_dwn_rgzj2.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(deg_nmj_sig_up_rgzj2.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
enr_dw_go = gp.enrichr(deg_nmj_sig_dwn_rgzj2.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

#gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Upregulated by aging", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Downregulated by aging in NMJ", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Upregulated by aging in NMJ", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Downregulated by aging in NMJ", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

In [ ]:
#print("\033[1m" + 'Regulated by DR: '+ "\033[0m" , deg_nmj_sig_rgzj4)
#print("\033[1m" + 'Upregulated by DR: '+ "\033[0m" , deg_nmj_sig_up_rgzj4)
#print("\033[1m" + 'Downregulated by DR: '+ "\033[0m" , deg_nmj_sig_dwn_rgzj4)
deg_nmj_sig_dwn_rgzj4
#print("\033[1m" + 'Regulated by combi: '+ "\033[0m" , deg_nmj_sig_rgzj5)

In [ ]:
enr_up = gp.enrichr(deg_nmj_sig_up_rgzj4.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_dw = gp.enrichr(deg_nmj_sig_dwn_rgzj4.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

enr_up_go = gp.enrichr(deg_nmj_sig_up_rgzj4.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
enr_dw_go = gp.enrichr(deg_nmj_sig_dwn_rgzj4.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

# trim (go:...)
enr_up_go.res2d.Term = enr_up_go.res2d.Term.str.split(" \(GO").str[0]
enr_dw_go.res2d.Term = enr_dw_go.res2d.Term.str.split(" \(GO").str[0]

gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Ercc1d/-, Upregulated by DR in NMJ", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"KEGG - Ercc1d/-, Downregulated by DR in NMJ", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

gp.dotplot(enr_up_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Ercc1d/-, Upregulated by DR in NMJ", cmap=plt.cm.autumn_r, cutoff = 0.5)
gp.dotplot(enr_dw_go.res2d, figsize=(3, 5), top_term=20, title=f"GO BP - Ercc1d/-, Downregulated by DR in NMJ", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)

In [ ]:
# Loop over the dataframes
for df_name in podo_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_PODO, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(50, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_PODO, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(50, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)

    enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), top_term=20, title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), top_term=20, title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        #gsplot.dotplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
        dotplot(enr_full.results, top_term=20, title=f"{df_short_name} KEGG all", cmap='RdBu_r', cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'])
    plt.show()

In [ ]:
# Map mouse to human genes
hum_ann_adata = translate(adata_myo)

In [ ]:
# Get the unique samples
samples = hum_ann_adata.obs['sample'].unique()

# Get the reference sample
reference_sample = hum_ann_adata[hum_ann_adata.obs['sample'] == 'SG18']

# Loop over the samples
for sample in samples:
    if sample == 'SG18':  # Skip the reference sample
        continue

    # Get the current sample
    current_sample = hum_ann_adata[hum_ann_adata.obs['sample'] == sample]
    
    # Concatenate the current sample with the reference sample
    sample_data = current_sample.concatenate(reference_sample)
    
    # Save metadata 
    meta = sample_data.obs
    meta = meta[['celltype']]
    # Set Index Name
    meta.index.name='Cell'

    # Save metadata and counts as CSV
    meta.to_csv(f'./cellphonedb//meta_{sample}.txt', sep = '\t')
    # Save counts as h5ad
    sample_data.write(f'./cellphonedb/counts_{sample}.h5ad')

    # Run the CellPhoneDB analysis
    deconvoluted, means, pvalues, significant_means = cpdb_statistical_analysis_method.call(
            cpdb_file_path = cpdb, 
            meta_file_path = f'./cellphonedb/meta_{sample}.txt',
            counts_file_path = f'./cellphonedb/counts_{sample}.h5ad',
            counts_data = 'hgnc_symbol',   # specify that you are using gene symbols
           # counts_data = 'ensembl',   
            output_path = f'./cellphonedb/{sample}_vs_SG18')

    # Rest of the code remains the same
    search_results = search_utils.search_analysis_results(
        query_cell_types_1 = ['PODO', 'vSMC/MC'],  # List of cells 1, will be paired to cells 2 (list or 'All').
        query_cell_types_2 = ['vSMC/MC', 'PODO'],     # List of cells 2, will be paired to cells 1 (list or 'All').
       # query_genes = ['TGFBR1'],                                       # filter interactions based on the genes participating (list).
     #   query_interactions = ['CSF1_CSF1R'],                            # filter intereactions based on their name (list).
        significant_means = significant_means,                          # significant_means file generated by CellPhoneDB.
        deconvoluted = deconvoluted,                                    # devonvoluted file generated by CellPhoneDB.
        separator = '|',                                                # separator (default: |) employed to split cells (cellA|cellB).
        long_format = True                                              # converts the output into a wide table, removing non-significant interactions
    )
    
    dotplot = kpy.plot_cpdb(
        adata = sample_data,
        cell_type1 = "PODO|vSMC/MC|PECs?|PT-2|IMM",
        # include EC-1(gEC)
        cell_type2 = "PODO|vSMC/MC|PECs?|PT-1", 
        means = means,
        pvals = pvalues,
        celltype_key = "celltype",
     #   genes = ["TGFB2", "CSF1R"],
        figsize = (15,5),
        title = f"Interactions vs. SG18 between PODO, vSMC/MC, PEC, PT-1, PT-2, IMM for {sample}",
        max_size = 6,
        highlight_size = 0.75,
        standard_scale = True
    )
    dotplot.save(f'./cellphonedb//{sample}_vs_SG18/{sample}_dotplot.png')
    
    # Generate plots for the current sample
    heatmap = kpy.plot_cpdb_heatmap(
            adata = sample_data,
            pvals = pvalues,
            celltype_key = "celltype",
            figsize = (5,5),
            title = f"Number of significant interactions vs. SG18 for {sample}",
            symmetrical = False,
        )
    heatmap.figure.savefig(f'./cellphonedb//{sample}_vs_SG18/{sample}_heatmap.png')

    # First create an empty graph
    G = nx.Graph()

    # Add nodes
    for gene in pd.concat([significant_means['gene_a'], significant_means['gene_b']]).unique():
        G.add_node(gene)

    # Add edges
    for i, row in significant_means.iterrows():
        G.add_edge(row['gene_a'], row['gene_b'])

    # Draw the graph
    plt.figure(figsize=(10,10))  # Set the figure size (optional)
    nx.draw(G, with_labels=True)
    plt.savefig(f'./cellphonedb/cellphonedb/{sample}_vs_SG18/{sample}_graph.png')
    plt.close()

In [ ]:
# Get the unique samples
samples = hum_ann_adata.obs['sample'].unique()

# Get the reference sample
reference_sample = hum_ann_adata[hum_ann_adata.obs['sample'] == 'SG20']

# Loop over the samples
for sample in samples:
    if sample == 'SG20':  # Skip the reference sample
        continue

    # Get the current sample
    current_sample = hum_ann_adata[hum_ann_adata.obs['sample'] == sample]
    
    # Concatenate the current sample with the reference sample
    sample_data = current_sample.concatenate(reference_sample)
    
    # Save metadata 
    meta = sample_data.obs
    meta = meta[['celltype']]
    # Set Index Name
    meta.index.name='Cell'

    # Save metadata and counts as CSV
    meta.to_csv(f'./cellphonedb/meta_{sample}.txt', sep = '\t')
    # Save counts as h5ad
    sample_data.write(f'./cellphonedb/counts_{sample}.h5ad')

    # Run the CellPhoneDB analysis
    deconvoluted, means, pvalues, significant_means = cpdb_statistical_analysis_method.call(
            cpdb_file_path = cpdb, 
            meta_file_path = f'./cellphonedb/meta_{sample}.txt',
            counts_file_path = f'./cellphonedb/counts_{sample}.h5ad',
            counts_data = 'hgnc_symbol',   # specify that you are using gene symbols
           # counts_data = 'ensembl',   
            output_path = f'./cellphonedb/{sample}_vs_SG20')

    # Rest of the code remains the same
    search_results = search_utils.search_analysis_results(
        query_cell_types_1 = ['PODO', 'vSMC/MC'],  # List of cells 1, will be paired to cells 2 (list or 'All').
        query_cell_types_2 = ['vSMC/MC', 'PODO'],     # List of cells 2, will be paired to cells 1 (list or 'All').
       # query_genes = ['TGFBR1'],                                       # filter interactions based on the genes participating (list).
     #   query_interactions = ['CSF1_CSF1R'],                            # filter intereactions based on their name (list).
        significant_means = significant_means,                          # significant_means file generated by CellPhoneDB.
        deconvoluted = deconvoluted,                                    # devonvoluted file generated by CellPhoneDB.
        separator = '|',                                                # separator (default: |) employed to split cells (cellA|cellB).
        long_format = True                                              # converts the output into a wide table, removing non-significant interactions
    )
    
    dotplot = kpy.plot_cpdb(
        adata = sample_data,
        cell_type1 = "PODO|vSMC/MC|PECs?|PT-2|IMM",
        # include EC-1(gEC)
        cell_type2 = "PODO|vSMC/MC|PECs?|PT-1", 
        means = means,
        pvals = pvalues,
        celltype_key = "celltype",
     #   genes = ["TGFB2", "CSF1R"],
        figsize = (15,5),
        title = f"Interactions vs. SG20 between PODO, vSMC/MC, PEC, PT-1, PT-2, IMM for {sample}",
        max_size = 6,
        highlight_size = 0.75,
        standard_scale = True
    )
    dotplot.save(f'./cellphonedb/{sample}_vs_SG20/{sample}_dotplot.png')
    
    # Generate plots for the current sample
    heatmap = kpy.plot_cpdb_heatmap(
            adata = sample_data,
            pvals = pvalues,
            celltype_key = "celltype",
            figsize = (5,5),
            title = f"Number of significant interactions vs. SG20 for {sample}",
            symmetrical = False,
        )
    heatmap.figure.savefig(f'./cellphonedb/{sample}_vs_SG20/{sample}_heatmap.png')
    
    # First create an empty graph
    G = nx.Graph()

    # Add nodes
    for gene in pd.concat([significant_means['gene_a'], significant_means['gene_b']]).unique():
        G.add_node(gene)

    # Add edges
    for i, row in significant_means.iterrows():
        G.add_edge(row['gene_a'], row['gene_b'])

    # Draw the graph
    plt.figure(figsize=(10,10))  # Set the figure size (optional)
    nx.draw(G, with_labels=True)
    plt.savefig(f'./cellphonedb/{sample}_vs_SG20/{sample}_graph.png')
    plt.close()

In [ ]:
# Concatenate gene_a and gene_b columns
all_genes = pd.concat([significant_means['gene_a'], significant_means['gene_b']])

# Count the occurrences of each gene
gene_counts = all_genes.value_counts()

# Plot a bar chart
gene_counts[:20].plot(kind='bar')

plt.ylabel('Number of Interactions')
plt.title('Top 20 genes with most interactions')

# Rotate x-axis labels
plt.xticks(rotation=45, ha="right")  # Rotate labels by 45 degrees and align them right

# Add some space at the bottom of the plot
plt.tight_layout()

plt.show()

In [ ]:
# Get top 20 genes with most interactions
top_genes = gene_counts[:20].index

# Filter significant_means for rows where either gene_a or gene_b is in top_genes
top_interactions = significant_means[(significant_means['gene_a'].isin(top_genes)) | (significant_means['gene_b'].isin(top_genes))]

# Keep only the cell-type columns and compute the mean interaction score for each gene
interaction_heatmap = top_interactions.groupby('gene_a').mean()

# Plot a heatmap
plt.figure(figsize=(25,9))
sns.heatmap(interaction_heatmap, cmap="viridis")
plt.xticks(rotation=90, ha="right")  # Rotate x-axis labels 
#plt.tight_layout()  # Adjust the layout
plt.subplots_adjust(bottom=0.25) 

plt.show()

In [ ]:
# List of cell types to include
cell_types_to_include = ["PODO", "vSMC/MC", "PECs?", "PT-2", "IMM", "EC-1(gEC)"]

# Generate all combinations of cell types
cell_type_combinations = list(itertools.combinations(cell_types_to_include, 2))

# Convert tuples to strings with '|' separator
cell_type_combinations = ['|'.join(combination) for combination in cell_type_combinations]

# Filter significant_means for rows where either gene_a or gene_b is in top_genes
top_interactions = significant_means[(significant_means['gene_a'].isin(top_genes)) | (significant_means['gene_b'].isin(top_genes))]

# Group by gene_a and calculate the mean interaction score, then filter columns to include only selected cell types
interaction_heatmap = top_interactions.groupby('gene_a').mean()[cell_type_combinations]

# Plot a heatmap
plt.figure(figsize=(25,9))
sns.heatmap(interaction_heatmap, cmap="viridis")
plt.xticks(rotation=90, ha="right")  # Rotate x-axis labels 
plt.subplots_adjust(bottom=0.25) 

plt.show()